# M2SVid Lightning full inference

Full pipeline: DepthCrafter → warping → M2SVid. Assumes source tree at `/teamspace/studios/this_studio`.

## 0. Check paths and GPU

In [ ]:
from pathlib import Path
import os, time, shutil, subprocess, json
STUDIO_ROOT = Path('/teamspace/studios/this_studio')
CODE_ROOT = STUDIO_ROOT
DATA_ROOT = STUDIO_ROOT / 'm2svid_demo'
CKPT_DIR = DATA_ROOT / 'ckpts'
OUTPUT_ARCHIVE = DATA_ROOT / 'outputs'
for p in [DATA_ROOT, CKPT_DIR, OUTPUT_ARCHIVE]: p.mkdir(parents=True, exist_ok=True)
os.chdir(CODE_ROOT)
print('CODE_ROOT=', CODE_ROOT)
print('README exists:', (CODE_ROOT/'README.md').exists())
print('demo exists:', (CODE_ROOT/'demo').exists())
!pwd
!nvidia-smi
!find demo -maxdepth 3 -type f | sort | head -50

## 0.5. Install/check system video tools


In [ ]:
# System-level video tools required by ffmpeg-python
# Lightning allows sudo apt-get in Studio. Safe to rerun.
!which ffmpeg || (sudo apt-get update -qq && sudo apt-get install -y ffmpeg)
!which ffprobe || true
!ffmpeg -version | head -1


## 1. Settings

In [ ]:
USE_OFFICIAL_DEMO = True
INPUT_DIR = DATA_ROOT/'inputs'; INPUT_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_INPUT_VIDEO = INPUT_DIR/'my_clip.mp4'
MAX_RES = 512
DEPTH_STEPS = 25
DISPARITY_PERC = 0.05
EXP_NAME = time.strftime('full_%Y%m%d_%H%M%S')
print(EXP_NAME)

## 2. Download/link checkpoints

In [ ]:
os.chdir(CODE_ROOT)
!python -m pip -q install gdown
if not (CKPT_DIR/'m2svid_weights.pt').exists():
    !wget -c -O {CKPT_DIR/'m2svid_weights.pt'} https://storage.googleapis.com/gresearch/m2svid/m2svid_weights.pt
else:
    print('M2SVid weight exists:', CKPT_DIR/'m2svid_weights.pt')
hi3d_zip = CKPT_DIR/'hi3d_ckpts.zip'
if not hi3d_zip.exists() and not (CKPT_DIR/'open_clip_pytorch_model.bin').exists() and not (CODE_ROOT/'ckpts/open_clip_pytorch_model.bin').exists():
    !python -m gdown 1j_NEG2CPhFeRetYziWK6Qe62R5h7lG_V -O {hi3d_zip}
else:
    print('Hi3D/OpenCLIP already present or zip exists')
!mkdir -p {CODE_ROOT/'ckpts'}
if hi3d_zip.exists() and not (CODE_ROOT/'ckpts/open_clip_pytorch_model.bin').exists():
    !unzip -n {hi3d_zip} -d {CODE_ROOT}
if (CKPT_DIR/'open_clip_pytorch_model.bin').exists():
    !ln -sf {CKPT_DIR/'open_clip_pytorch_model.bin'} {CODE_ROOT/'ckpts/open_clip_pytorch_model.bin'}
!ln -sf {CKPT_DIR/'m2svid_weights.pt'} {CODE_ROOT/'ckpts/m2svid_weights.pt'}
!find ckpts -maxdepth 2 -type f -o -type l | sort | head -80

## 3. Install M2SVid deps

In [ ]:
os.chdir(CODE_ROOT)
!python -m pip -q install omegaconf einops pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 opencv-python-headless==4.10.0.84 omegaconf==2.3.0 einops==0.8.0 pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 pytorch-msssim==1.0.0 transformers diffusers accelerate tokenizers sentencepiece ftfy ffmpeg-python av imageio imageio-ffmpeg safetensors scipy scikit-image pillow tqdm matplotlib pandas numpy==1.26.4 xformers || true

# Check from the current notebook kernel instead of using a shell heredoc.
# In IPython, `!python - <<'PY' ... PY` can be parsed incorrectly line-by-line.
import torch, torchvision
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('torchvision', torchvision.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 4. Prepare DepthCrafter

In [ ]:
os.chdir(CODE_ROOT)
if not (CODE_ROOT/'third_party/DepthCrafter').exists():
    !mkdir -p third_party
    !git clone --depth 1 https://github.com/Tencent/DepthCrafter.git third_party/DepthCrafter

# DepthCrafter in this bundled tree may not ship a requirements.txt, so install
# the runtime deps explicitly. Safe to rerun.
!python -m pip install -q   fire   decord   opencv-python-headless==4.10.0.84   matplotlib   scipy   einops   diffusers   transformers   accelerate   safetensors   huggingface-hub   imageio   imageio-ffmpeg

%cd /teamspace/studios/this_studio/third_party/DepthCrafter
if Path('requirements.txt').exists() and Path('requirements.txt').stat().st_size > 0:
    !python -m pip -q install -r requirements.txt || true
else:
    print('No non-empty DepthCrafter requirements.txt; using explicit deps above')
%cd /teamspace/studios/this_studio

# Import sanity check before running the expensive depth job.
!python - <<'PY'
import fire, decord, cv2, torch
import diffusers, transformers, accelerate
print('fire OK')
print('decord OK')
print('cv2', cv2.__version__)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
PY

!test -f third_party/DepthCrafter/run.py && echo 'DepthCrafter run.py found' || find third_party/DepthCrafter -maxdepth 2 -type f | head -80


## 5. Prepare input

In [ ]:
os.chdir(CODE_ROOT)
INPUT_VIDEO = CODE_ROOT/'demo/input.mp4' if USE_OFFICIAL_DEMO else CUSTOM_INPUT_VIDEO
assert INPUT_VIDEO.exists(), f'Missing input: {INPUT_VIDEO}'
WORK_INPUT = CODE_ROOT/'inputs/input.mp4'; WORK_INPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT_VIDEO, WORK_INPUT)
print('Using', INPUT_VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,duration -of default=nw=1 {WORK_INPUT}

## 6. Run DepthCrafter

In [ ]:
os.chdir(CODE_ROOT)
out=CODE_ROOT/'outputs/depthcrafter_lightning_full'
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
start=time.time()
cmd=f'''PYTHONPATH="third_party/DepthCrafter/::${{PYTHONPATH}}" python third_party/DepthCrafter/run.py --video-path {WORK_INPUT} --save_folder outputs/depthcrafter_lightning_full --save_npz True --num_inference_steps {DEPTH_STEPS} --max_res {MAX_RES}'''
print(cmd); ret=subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed:', round(time.time()-start,1))
!find outputs/depthcrafter_lightning_full -maxdepth 2 -type f -print
!nvidia-smi

## 7. Run warping

In [ ]:
candidates=sorted((CODE_ROOT/'outputs/depthcrafter_lightning_full').glob('*.npz'))
print(candidates); assert candidates, 'No depth npz found'
DEPTH_NPZ=candidates[0]
os.chdir(CODE_ROOT)
rep=CODE_ROOT/'outputs/reprojected_lightning_full'
if rep.exists(): shutil.rmtree(rep)
rep.mkdir(parents=True, exist_ok=True)
start=time.time()
cmd=f'''PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${{PYTHONPATH}}" python warping.py --video_path {WORK_INPUT} --depth_path {DEPTH_NPZ} --output_path_reprojected outputs/reprojected_lightning_full/input_reprojected.mp4 --output_path_mask outputs/reprojected_lightning_full/input_reprojected_mask.mp4 --disparity_perc {DISPARITY_PERC}'''
print(cmd); ret=subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed:', round(time.time()-start,1))
!find outputs/reprojected_lightning_full -type f -print

## 8. Run M2SVid refinement

In [ ]:
os.chdir(CODE_ROOT)
out=CODE_ROOT/'outputs/m2svid_lightning_full'
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
start=time.time()
cmd='''PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${PYTHONPATH}" python inpaint_and_refine.py \
--mask_antialias 0 \
--model_config configs/m2svid.yaml \
--ckpt ckpts/m2svid_weights.pt \
--video_path inputs/input.mp4 \
--reprojected_path outputs/reprojected_lightning_full/input_reprojected.mp4 \
--reprojected_mask_path outputs/reprojected_lightning_full/input_reprojected_mask.mp4 \
--output_folder outputs/m2svid_lightning_full'''
print(cmd); ret=subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed:', round(time.time()-start,1))
!find outputs/m2svid_lightning_full -type f -print
!nvidia-smi

## 9. View and archive

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path

def show_video(path, width=720):
    path = Path(path)
    assert path.exists(), f'Missing {path}'
    data = 'data:video/mp4;base64,' + b64encode(path.read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="{data}" type="video/mp4"></video><p><code>{path}</code></p>'))

show_video(CODE_ROOT/'inputs/input.mp4', 520)
show_video(CODE_ROOT/'outputs/reprojected_lightning_full/input_reprojected.mp4', 520)
show_video(CODE_ROOT/'outputs/reprojected_lightning_full/input_reprojected_mask.mp4', 520)
if (CODE_ROOT/'outputs/m2svid_lightning_full/input_sbs.mp4').exists(): show_video(CODE_ROOT/'outputs/m2svid_lightning_full/input_sbs.mp4', 720)
archive=OUTPUT_ARCHIVE/EXP_NAME; archive.mkdir(parents=True, exist_ok=True)
for rel in ['inputs','outputs/depthcrafter_lightning_full','outputs/reprojected_lightning_full','outputs/m2svid_lightning_full']:
    src=CODE_ROOT/rel
    if src.exists():
        dst=archive/rel.replace('/','_')
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src,dst)
print('Archived to:', archive)